# Confidence-Routed Council (Dynamic Ensemble)

This notebook implements a Two-Stage Ensemble architecture to beat the 0.7708 Weighted F1 score.

- **Stage 1**: Fast Baseline (Scene-batching with Plain Prompt + Confidence score).
- **Router**: Filters predictions where `confidence < 0.80`.
- **Stage 2**: Deep-Dive Specialist (Runs the Unified Prompt + Bio Cards *only* on uncertain utterances).
- **Final**: Merges and evaluates.

In [1]:
import os
import json
import re
from datetime import datetime
from pathlib import Path
from typing import Optional, Dict, Any

import pandas as pd
import sys
sys.path.append("../")

import vertexai
from dotenv import load_dotenv
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from tqdm import tqdm

load_dotenv()

# Configuration
BASE_DIR = "../"
DATA_PATH = os.path.join(BASE_DIR, "data", "test_sent_emo.csv")
BIO_CARDS_PATH = os.path.join(BASE_DIR, "logs", "speaker_bio_cards.json")
OUTPUT_DIR = os.path.join(BASE_DIR, "logs", "ensemble_router")
os.makedirs(OUTPUT_DIR, exist_ok=True)

PROJECT_ID = os.getenv("LLAMA_MODEL_PROJECT_ID") or os.getenv("TUNED_MODEL_PROJECT_ID")
LOCATION = os.getenv("VERTEX_LOCATION", "us-central1")
ENDPOINT_ID = os.getenv("LLAMA31_ENDPOINT_ID", "2346569469662330880")

if not PROJECT_ID:
    raise ValueError("Missing project id. Set LLAMA_MODEL_PROJECT_ID or TUNED_MODEL_PROJECT_ID in .env")

vertexai.init(project=PROJECT_ID, location=LOCATION)
from vertexai.generative_models import GenerativeModel

llama31_model = GenerativeModel(f"projects/{PROJECT_ID}/locations/{LOCATION}/endpoints/{ENDPOINT_ID}")

print(f"✅ Configuration Complete")
print(f"  Project: {PROJECT_ID}")
print(f"  Location: {LOCATION}")
print(f"  Endpoint: {ENDPOINT_ID}")
print(f"  Output: {OUTPUT_DIR}")

✅ Configuration Complete
  Project: project-77549f95-0391-4b29-911
  Location: us-central1
  Endpoint: 2346569469662330880
  Output: ../logs\ensemble_router


## Load Data & Utilities

In [2]:
def load_data_from_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = df.sort_values(["Dialogue_ID", "Utterance_ID"]).reset_index(drop=True)
    return df

def load_speaker_bio_cards(json_file: str = BIO_CARDS_PATH) -> Dict[str, Any]:
    try:
        with open(json_file, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception as e:
        print(f"⚠️  Bio cards load error: {e}")
        return {}

def format_bio_cards_context(bio_cards: Dict, speakers: list) -> str:
    if not bio_cards or not speakers:
        return ""
    context = "\n### SPEAKER PROFILES\n"
    for speaker in speakers:
        if speaker in bio_cards:
            bio = bio_cards[speaker]
            bio_text = "\n".join([f"{k}: {v}" for k, v in bio.items()]) if isinstance(bio, dict) else str(bio)
            context += f"\n**{speaker}:**\n{bio_text}\n"
    return context

print("Loading dataset...")
df = load_data_from_csv(Path(DATA_PATH))
print("Loading speaker bio cards...")
bio_cards = load_speaker_bio_cards()
print(f"\n✅ Data Loaded: {len(df)} rows")

Loading dataset...
Loading speaker bio cards...

✅ Data Loaded: 2610 rows


## Stage 1: The Fast Baseline

In [3]:
STAGE1_INSTRUCTION = (
    """
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are an expert Emotion Recognition assistant specialized in the MELD benchmark.
For each scene, predict one emotion for EVERY utterance using labels: [anger, disgust, fear, joy, neutral, sadness, surprise].
Return JSON only in this schema:
{
  "predictions": [
    {
        "utterance_id": "id", 
        "predicted_emotion": "label", 
        "confidence": 0.0,
        "reasoning": "short reason"
    }
  ]
}
The 'confidence' field must be a float between 0.0 and 1.0 representing your certainty.
<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""
)

def call_llama(prompt: str) -> str:
    response = llama31_model.generate_content(prompt)
    return response.text

def parse_predictions(text: str) -> dict:
    if not text: return {}
    parsed = None
    try:
        match = re.search(r"\{[\s\S]*\}", text)
        if match: parsed = json.loads(match.group(0))
    except Exception: pass
    if not parsed: return {}
    
    predictions = parsed.get("predictions", []) if isinstance(parsed, dict) else (parsed if isinstance(parsed, list) else [])
    
    out = {}
    for item in predictions:
        if not isinstance(item, dict): continue
        utt_id = item.get("utterance_id")
        if utt_id is None: continue
        emotion = item.get("predicted_emotion") or item.get("emotion")
        confidence = item.get("confidence", 0.5)
        out[str(utt_id)] = {
            "predicted_emotion": str(emotion).lower().strip() if emotion else None,
            "confidence": float(confidence) if isinstance(confidence, (int, float)) else 0.5,
            "reasoning": str(item.get("reasoning", ""))
        }
    return out

def build_stage1_batches(df: pd.DataFrame) -> list:
    scenes = []
    for dialogue_id, group in df.groupby("Dialogue_ID", sort=False):
        scene_rows = []
        scene_lines = []
        for _, row in group.iterrows():
            scene_rows.append(row.to_dict())
            scene_lines.append(f"{row['Utterance_ID']} | {row['Speaker']}: {row['Utterance']}")
        
        scene_prompt = (
            f"{STAGE1_INSTRUCTION}\n\n"
            f"Scene Dialogue ID: {dialogue_id}\n"
            "Utterances in order:\n"
            + "\n".join(scene_lines)
            + "\n\nReturn predictions for ALL utterance_id values above."
        )
        scenes.append({"dialogue_id": dialogue_id, "rows": scene_rows, "prompt": scene_prompt})
    return scenes

def run_stage1(df: pd.DataFrame, limit: Optional[int] = None):
    scenes = build_stage1_batches(df)
    if limit is not None: scenes = scenes[:limit]
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = os.path.join(OUTPUT_DIR, f"stage1_predictions_{timestamp}.csv")
    
    records = []
    print(f"\n🚀 Running Stage 1 Inference on {len(scenes)} scenes...")
    
    for scene in tqdm(scenes, desc="Stage 1"):
        model_output = call_llama(scene["prompt"])
        pred_map = parse_predictions(model_output)
        
        for row in scene["rows"]:
            pred_item = pred_map.get(str(row["Utterance_ID"]), {})
            record = row.copy()
            record["stage1_emotion"] = pred_item.get("predicted_emotion")
            record["stage1_confidence"] = pred_item.get("confidence")
            record["stage1_reasoning"] = pred_item.get("reasoning", "")
            record["stage1_raw"] = model_output
            records.append(record)
            
    res_df = pd.DataFrame(records)
    res_df.to_csv(output_path, index=False)
    print(f"✅ Stage 1 complete! Saved to {output_path}")
    return res_df

## Run Stage 1

In [4]:
# Run Stage 1
# limit = None for full run, limit = 5 for quick testing
LIMIT = None
stage1_df = run_stage1(df, limit=LIMIT)



🚀 Running Stage 1 Inference on 280 scenes...


Stage 1:   1%|          | 3/280 [00:40<1:01:51, 13.40s/it]


KeyboardInterrupt: 

## The Router & Stage 2: Deep-Dive Specialist

In [ ]:
UNIFIED_SYSTEM_PROMPT = """
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are an EXPERT EMOTION RECOGNITION SYSTEM specialized in the MELD benchmark. 
Your task is to predict ONE emotion for a SPECIFIC utterance using ONLY these labels:
[anger, disgust, fear, joy, neutral, sadness, surprise]

You will analyze the utterance through 7 integrated analytical lenses:
1. Character Behavioral Baseline (using provided profiles)
2. Scene Vibe & Emotional Arc
3. Temporal Dynamics & Shifts
4. Linguistic Pragmatics & Subtext
5. Relational Dynamics & Safety
6. Social Face Management
7. Final Synthesis

FINAL OUTPUT FORMAT (STRICT JSON):
{
  "predictions": [
    {
      "utterance_id": "exact_id_from_input",
      "predicted_emotion": "single_label_only",
      "confidence": 0.0,
      "reasoning": "2-sentence Chain-of-Thought explaining choice over alternatives"
    }
  ]
}
<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

def build_stage2_prompt(target_row, group_df, bio_cards):
    target_idx = group_df.index[group_df['Utterance_ID'] == target_row['Utterance_ID']].tolist()[0]
    # Get 2 before, 1 after for context
    start_idx = max(0, target_idx - 2)
    end_idx = min(len(group_df), target_idx + 2)
    context_df = group_df.iloc[start_idx:end_idx]
    
    scene_lines = []
    for _, row in context_df.iterrows():
        prefix = "--> TARGET: " if row['Utterance_ID'] == target_row['Utterance_ID'] else "    CONTEXT: "
        scene_lines.append(f"{prefix}{row['Utterance_ID']} | {row['Speaker']}: {row['Utterance']}")
        
    bio_context = format_bio_cards_context(bio_cards, [target_row['Speaker']])
    
    user_content = (
        f"Dialogue ID: {target_row['Dialogue_ID']}\n"
        f"{bio_context}\n"
        f"### SCENE MICRO-CONTEXT\n"
        + "\n".join(scene_lines) + "\n\n"
        f"The baseline model previously predicted '{target_row.get('stage1_emotion', 'unknown')}' with low confidence.\n"
        f"Re-evaluate TARGET utterance_id '{target_row['Utterance_ID']}' using your 7-agent deep analysis."
    )
    return f"{UNIFIED_SYSTEM_PROMPT}\n\nUser Input:\n{user_content}"

def run_stage2_refinement(stage1_df: pd.DataFrame, threshold: float = 0.80):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = os.path.join(OUTPUT_DIR, f"final_predictions_{timestamp}.csv")
    
    # Identify hard utterances
    mask = (stage1_df['stage1_confidence'].isna()) | (stage1_df['stage1_confidence'] < threshold)
    hard_df = stage1_df[mask]
    
    print(f"\n🔍 Router selected {len(hard_df)} utterances for Stage 2 Refinement (Confidence < {threshold}).")
    
    final_records = stage1_df.copy()
    final_records['final_emotion'] = final_records['stage1_emotion']
    final_records['is_refined'] = False
    
    for idx, row in tqdm(hard_df.iterrows(), total=len(hard_df), desc="Stage 2 Refinement"):
        dialogue_id = row['Dialogue_ID']
        group_df = df[df['Dialogue_ID'] == dialogue_id].reset_index(drop=True)
        
        prompt = build_stage2_prompt(row, group_df, bio_cards)
        model_output = call_llama(prompt)
        pred_map = parse_predictions(model_output)
        
        pred_item = pred_map.get(str(row['Utterance_ID']))
        if pred_item and pred_item.get("predicted_emotion"):
            final_records.at[idx, 'final_emotion'] = pred_item["predicted_emotion"]
            final_records.at[idx, 'stage2_confidence'] = pred_item["confidence"]
            final_records.at[idx, 'stage2_reasoning'] = pred_item["reasoning"]
            final_records.at[idx, 'is_refined'] = True
            
    final_records.to_csv(output_path, index=False)
    print(f"✅ Ensemble complete! Saved to {output_path}")
    return final_records

## Run Router & Stage 2

In [ ]:
# Run Stage 2
THRESHOLD = 0.80
final_df = run_stage2_refinement(stage1_df, threshold=THRESHOLD)


## Evaluation

In [ ]:
def evaluate_predictions(pred_df, pred_column):
    valid_df = pred_df.dropna(subset=["Emotion", pred_column]).copy()
    valid_df["Emotion"] = valid_df["Emotion"].str.lower().str.strip()
    valid_df[pred_column] = valid_df[pred_column].str.lower().str.strip()
    
    all_emotions = sorted(set(list(valid_df["Emotion"].unique()) + list(valid_df[pred_column].unique())))
    
    print(f"\n{'='*80}")
    print(f"CLASSIFICATION REPORT FOR: {pred_column}")
    print(f"{'='*80}\n")
    print(classification_report(valid_df["Emotion"], valid_df[pred_column], labels=all_emotions, digits=4, zero_division=0))
    
    wf1 = f1_score(valid_df["Emotion"], valid_df[pred_column], average="weighted", zero_division=0)
    print(f"Weighted F1: {wf1:.4f}")

# Evaluate Stage 1 Alone
evaluate_predictions(final_df, "stage1_emotion")

# Evaluate Final Ensemble (Stage 1 + Stage 2)
evaluate_predictions(final_df, "final_emotion")

refined_count = final_df['is_refined'].sum()
print(f"\nTotal utterances refined in Stage 2: {refined_count}")
